> **Runtime Recommendation:** Use **T4 GPU** runtime in Google Colab for best results.
> TPU runtime is not recommended because TensorFlow's TPU support requires specific
> code modifications (e.g., `TPUStrategy`) and may not be available in all Colab
> environments. The T4 GPU provides 15 GB of VRAM and works seamlessly with standard
> TensorFlow/Keras code.
>
> To select: **Runtime → Change runtime type → T4 GPU**

# Network Intrusion Detection System using Transformer - CSE-CIC-IDS2018 Dataset (Improved)

This notebook is an improved version of `NIDS_Transformer_CICIDS2018.ipynb`. It applies the same
transformer-based multi-class classification methodology to the **CSE-CIC-IDS2018** dataset with
the following enhancements:

1. **Accuracy metrics** added to model compilation.
2. **Training callbacks**: `EarlyStopping` and `ReduceLROnPlateau` with increased max epochs.
3. **Proper 3-D input handling** in `TransformerModel.call` via `tf.expand_dims`.
4. **Class imbalance handling** using `sklearn` computed class weights.
5. **ROC-AUC evaluation** with One-vs-Rest multi-class ROC curves.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
import gc

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras import layers, Model

%matplotlib inline

## 1. Load CSE-CIC-IDS2018 Dataset

The CSE-CIC-IDS2018 dataset consists of multiple CSV files, one per day of captured traffic.
Update the `data_dir` path below to point to the folder containing the CSV files.

The dataset can be downloaded from:
- [CIC IDS 2018 on AWS](https://www.unb.ca/cic/datasets/ids-2018.html)
- [Kaggle CSE-CIC-IDS2018](https://www.kaggle.com/datasets/solarmainframe/ids-intrusion-csv)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Update this path to point to the folder containing CSE-CIC-IDS2018 CSV files
data_dir = '/content/drive/MyDrive/CSE-CIC-IDS2018/'

# --- Memory Configuration ---
# Rows processed per iteration; 50k keeps peak memory low on Colab free tier
CHUNK_SIZE = 50_000
# Maximum total rows to load (set to None to load everything).
# The full dataset has ~16M rows; 2M is a practical limit for Colab's ~12 GB RAM.
MAX_ROWS = 2_000_000

In [ ]:
csv_files = sorted(glob.glob(os.path.join(data_dir, '*.csv')))
print(f'Found {len(csv_files)} CSV files:')
for f in csv_files:
    print(f'  {os.path.basename(f)}')

def optimize_dtypes(df):
    """Downcast numeric columns to reduce memory usage."""
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    return df

# Load, clean, and optimize each chunk before accumulating.
# This keeps peak memory much lower than loading first and cleaning later.
cleaned_chunks = []
total_rows = 0
stop_loading = False

for f in csv_files:
    for chunk in pd.read_csv(f, low_memory=False, chunksize=CHUNK_SIZE):
        chunk.columns = chunk.columns.str.strip()
        # Clean: remove infinities and missing values per-chunk
        chunk.replace([np.inf, -np.inf], np.nan, inplace=True)
        chunk.dropna(inplace=True)
        # Remove duplicates within each chunk
        chunk.drop_duplicates(inplace=True)
        chunk = optimize_dtypes(chunk)
        cleaned_chunks.append(chunk)
        total_rows += len(chunk)
        if MAX_ROWS is not None and total_rows >= MAX_ROWS:
            stop_loading = True
            break
    print(f'Loaded {os.path.basename(f)}  (total rows so far: {total_rows:,})')
    if stop_loading:
        print(f'Reached MAX_ROWS limit ({MAX_ROWS:,}). Stopping.')
        break

data = pd.concat(cleaned_chunks, ignore_index=True)
del cleaned_chunks
gc.collect()

# Enforce the row limit after concatenation
if MAX_ROWS is not None and len(data) > MAX_ROWS:
    data = data.sample(n=MAX_ROWS, random_state=42).reset_index(drop=True)

print(f'\nDataset shape: {data.shape}')
print(f'Memory usage: {data.memory_usage(deep=True).sum() / 1e6:.1f} MB')

## 2. Data Cleaning

Most cleaning (inf removal, NaN dropping, per-chunk deduplication, dtype optimization)
was already performed during the chunked loading above.

In [ ]:
print('Columns:', data.columns.tolist())

# The label column in CSE-CIC-IDS2018 is typically named 'Label'
label_col = 'Label'
print(f'\nUnique labels ({data[label_col].nunique()}):')
print(data[label_col].value_counts())

## 3. Exploratory Data Analysis

In [ ]:
data.head()

In [ ]:
data.describe()

In [ ]:
label_counts = data[label_col].value_counts()
plt.figure(figsize=(12, 6))
label_counts.plot(kind='bar')
plt.title('Class Distribution - CSE-CIC-IDS2018')
plt.xlabel('Attack Type')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4. Feature and Label Preparation

In [ ]:
# Separate features and labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(data[label_col]).astype('int32')
class_names = label_encoder.classes_.tolist()
num_classes = len(class_names)

# Drop label and any non-numeric columns (e.g., Timestamp)
data.drop(columns=[label_col], inplace=True)
non_numeric_cols = data.select_dtypes(include=['object', 'datetime']).columns.tolist()
if non_numeric_cols:
    print(f'Dropping non-numeric columns: {non_numeric_cols}')
    data.drop(columns=non_numeric_cols, inplace=True)

print(f'Feature matrix shape: {data.shape}')
print(f'Number of classes: {num_classes}')
print(f'Classes: {class_names}')

## 5. Train / Validation / Test Split

In [ ]:
# Convert features to float32 numpy array and free the DataFrame
X = data.values.astype('float32')
del data
gc.collect()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
del X, y
gc.collect()

X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5, random_state=42)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape:   {X_val.shape},   y_val shape:   {y_val.shape}")
print(f"X_test shape:  {X_test.shape},  y_test shape:  {y_test.shape}")

In [ ]:
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train).astype('float32')
X_val = scaler.transform(X_val).astype('float32')
X_test = scaler.transform(X_test).astype('float32')
gc.collect()

## 6. Transformer Model Architecture

Using the same MultiHeadSelfAttention and TransformerModel as in `NIDS_Transformer_Multi_Class.ipynb`.

**Improvement:** The `TransformerModel.call` method now uses `tf.expand_dims` to reshape the
2-D tabular input `(batch_size, features)` into a 3-D tensor `(batch_size, 1, features)` so
that the attention and pooling layers operate on a proper sequence dimension.

In [ ]:
class MultiHeadSelfAttention(layers.Layer):
    def __init__(self, embed_dim, num_heads):
        super(MultiHeadSelfAttention, self).__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        assert embed_dim % num_heads == 0
        self.projection_dim = embed_dim // num_heads
        self.query_dense = layers.Dense(embed_dim)
        self.key_dense = layers.Dense(embed_dim)
        self.value_dense = layers.Dense(embed_dim)
        self.combine_heads = layers.Dense(embed_dim)

    def attention(self, query, key, value):
        score = tf.matmul(query, key, transpose_b=True)
        dim_key = tf.cast(tf.shape(key)[-1], tf.float32)
        scaled_score = score / tf.math.sqrt(dim_key)
        weights = tf.nn.softmax(scaled_score, axis=-1)
        output = tf.matmul(weights, value)
        return output, weights

    def separate_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.projection_dim))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        query = self.query_dense(inputs)
        key = self.key_dense(inputs)
        value = self.value_dense(inputs)
        query = self.separate_heads(query, batch_size)
        key = self.separate_heads(key, batch_size)
        value = self.separate_heads(value, batch_size)
        attention, weights = self.attention(query, key, value)
        attention = tf.transpose(attention, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(attention, (batch_size, -1, self.embed_dim))
        output = self.combine_heads(concat_attention)
        return output, weights


class TransformerModel(Model):
    def __init__(self, num_heads, ff_dim, num_layers, input_dim, output_dim, dropout_rate=0.1):
        super(TransformerModel, self).__init__()
        self.embedding_layer = layers.Dense(input_dim, activation='relu')
        self.attention = MultiHeadSelfAttention(input_dim, num_heads)
        self.ffn = tf.keras.Sequential(
            [layers.Dense(ff_dim, activation="relu"), layers.Dense(input_dim),]
        )
        self.dropout1 = layers.Dropout(dropout_rate)
        self.dropout2 = layers.Dropout(dropout_rate)
        self.layer_normal1 = layers.LayerNormalization(epsilon=1e-6)
        self.layer_normal2 = layers.LayerNormalization(epsilon=1e-6)
        self.global_average_pooling = layers.GlobalAveragePooling1D()
        self.fc = layers.Dense(output_dim, activation='softmax')

    def call(self, inputs):
        # Expand dims to treat tabular data as a sequence of length 1:
        # (batch_size, features) -> (batch_size, 1, features)
        x = tf.expand_dims(inputs, axis=1)
        x = self.embedding_layer(x)
        x = self.layer_normal1(x + self.dropout1(self.attention(x)[0]))
        x = self.layer_normal2(x + self.dropout2(self.ffn(x)))
        # GlobalAveragePooling1D reduces (batch_size, 1, features) -> (batch_size, features)
        x = self.global_average_pooling(x)
        return self.fc(x)

## 7. Class Imbalance Handling

**Improvement:** CSE-CIC-IDS2018 is heavily imbalanced (most traffic is benign). We compute
class weights using `sklearn` so that minority attack classes receive higher weight during
training, preventing the model from being biased towards the majority class.

In [ ]:
classes = np.unique(y_train)
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)
class_weights_dict = dict(zip(classes.tolist(), class_weights_array.tolist()))

print('Class weights:')
for cls_idx, weight in class_weights_dict.items():
    print(f'  Class {cls_idx} ({class_names[cls_idx]}): {weight:.4f}')

## 8. Model Training

**Improvements:**
- Model compiled with `metrics=['accuracy']` to track accuracy during training.
- `EarlyStopping` (patience=3) stops training when validation loss stops improving and
  restores the best weights automatically.
- `ReduceLROnPlateau` (patience=2, factor=0.5) halves the learning rate when validation
  loss plateaus, helping the model converge to a better minimum.
- Max epochs increased to 20 (early stopping will halt sooner if needed).
- Class weights passed to `fit` to address dataset imbalance.

In [ ]:
num_heads = 2
ff_dim = 128
num_layers = 2
input_dim = X_train.shape[1]
output_dim = num_classes
dropout_rate = 0.2

print(f'Input dimension: {input_dim}')
print(f'Output dimension (number of classes): {output_dim}')

model_trans_multi = TransformerModel(num_heads, ff_dim, num_layers, input_dim, output_dim)

# Use SparseCategoricalCrossentropy with integer labels to avoid one-hot memory overhead
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

model_trans_multi.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

In [ ]:
BATCH_SIZE = 64
# Buffer size for shuffling; trades memory for randomization quality
SHUFFLE_BUFFER_SIZE = 10_000

train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_dataset = train_dataset.shuffle(buffer_size=SHUFFLE_BUFFER_SIZE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Free the numpy arrays now that tf.data holds the data
del X_train, X_val, y_val
gc.collect()

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2
)

model_trans_multi.fit(
    train_dataset,
    epochs=20,
    validation_data=val_dataset,
    callbacks=[early_stopping, reduce_lr],
    class_weight=class_weights_dict
)

In [ ]:
history_df = pd.DataFrame(model_trans_multi.history.history)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_df['loss'], label='Train Loss')
axes[0].plot(history_df['val_loss'], label='Val Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(history_df['accuracy'], label='Train Accuracy')
axes[1].plot(history_df['val_accuracy'], label='Val Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Evaluation

In [ ]:
test_dataset = tf.data.Dataset.from_tensor_slices(X_test).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

y_pred_prob = model_trans_multi.predict(test_dataset)
y_pred = np.argmax(y_pred_prob, axis=-1)

print(f'y_pred shape: {y_pred.shape}')
print(f'y_test shape: {y_test.shape}')

In [ ]:
print(classification_report(y_test, y_pred, digits=4, target_names=class_names))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(14, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names)
plt.title('Confusion Matrix - CSE-CIC-IDS2018 Multi-Class Classification')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 10. ROC-AUC Evaluation

**Improvement:** ROC curves are plotted using a One-vs-Rest (OvR) strategy for each class.
The micro-average ROC-AUC provides an aggregate measure across all classes.

In [ ]:
# Binarize ground-truth labels for One-vs-Rest ROC computation
y_test_bin = label_binarize(y_test, classes=list(range(num_classes)))

# Compute ROC curve and AUC for each class
fpr = {}
tpr = {}
roc_auc = {}

for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_pred_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Micro-average ROC curve (aggregate over all classes)
fpr['micro'], tpr['micro'], _ = roc_curve(y_test_bin.ravel(), y_pred_prob.ravel())
roc_auc['micro'] = auc(fpr['micro'], tpr['micro'])

print(f'Micro-average ROC-AUC: {roc_auc["micro"]:.4f}')
for i, name in enumerate(class_names):
    print(f'  {name}: AUC = {roc_auc[i]:.4f}')

In [ ]:
plt.figure(figsize=(12, 8))

# Plot micro-average ROC curve
plt.plot(
    fpr['micro'], tpr['micro'],
    label=f'Micro-average (AUC = {roc_auc["micro"]:.4f})',
    color='navy', linestyle=':', linewidth=2
)

# Plot per-class ROC curves
colors = plt.cm.tab20.colors
for i, name in enumerate(class_names):
    plt.plot(
        fpr[i], tpr[i],
        color=colors[i % len(colors)],
        label=f'{name} (AUC = {roc_auc[i]:.4f})'
    )

plt.plot([0, 1], [0, 1], 'k--', linewidth=1)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC-AUC Curves - CSE-CIC-IDS2018 (One-vs-Rest)')
plt.legend(loc='lower right', fontsize='small')
plt.tight_layout()
plt.show()

# Free large arrays no longer needed
del y_pred_prob, test_dataset, X_test
gc.collect()